# Week 1 (starter): Tokenization Analysis

This is the starter notebook for the Week 1 assignment. It runs as-is on placeholder text so you can see the shape of each step; your job is to replace the placeholders with your own passages and analysis, then commit it to your repository and open a pull request.

Cells marked **TODO (you)** are where you do the work. Everything runs in Jupyter or Google Colab. No GPU, no API key, one dependency: `tiktoken`.

The five parts match the assignment: stand up your repo, run the analysis, evaluate with real figures, find one failure, and submit.

In [19]:
# Setup. In Colab, uncomment the install line on first run.
!pip install tiktoken
import os, pathlib
os.environ['TIKTOKEN_CACHE_DIR'] = str((pathlib.Path('.') / '.tiktoken_cache').resolve())
os.makedirs(os.environ['TIKTOKEN_CACHE_DIR'], exist_ok=True)

import tiktoken
gpt4  = tiktoken.get_encoding('cl100k_base')   # GPT-4 / GPT-3.5
gpt4o = tiktoken.get_encoding('o200k_base')    # GPT-4o
print('tiktoken', tiktoken.__version__, '- encoders ready (cl100k_base, o200k_base)')

tiktoken 0.14.0 - encoders ready (cl100k_base, o200k_base)


## Part 1: Stand up your repository

Do this once, outside the notebook:

1. Create a public repo (suggested name `cosc-650`).
2. Add a `README.md` a stranger could read (what it is, how it is organized, the tools you use).
3. Add an agent context file that your AI tool reads, with project context and conventions. `AGENTS.md` is the cross-tool convention; `CLAUDE.md` and `GEMINI.md` are tool-specific variants. Use whichever your tool reads.
4. Work on a branch and open a pull request into `main`. You will do this every week.

Then commit this notebook into the repo and keep going.

## Helpers (provided)

Two small functions: count tokens for a string, and show the exact sub-token pieces a word breaks into. The demo uses a line you may recognize.

In [20]:
def count_tokens(text, enc):
    return len(enc.encode(text))

def show_split(word, enc=gpt4):
    ids = enc.encode(word)
    pieces = [enc.decode([i]) for i in ids]
    print(f'{word!r:18s} -> {len(ids)} token(s): {pieces}')

# demo: some short strings are a single token; capitalized or rarer words fragment
for w in ['Panic', ' towel', '42', 'antidisestablishmentarianism']:
    show_split(w)

'Panic'            -> 2 token(s): ['P', 'anic']
' towel'           -> 1 token(s): [' towel']
'42'               -> 1 token(s): ['42']
'antidisestablishmentarianism' -> 6 token(s): ['ant', 'idis', 'establish', 'ment', 'arian', 'ism']


## Part 2: Your passages

**TODO (you):** replace the two placeholders with your own text. The non-English passage must be at least 100 words, with a faithful English translation. The placeholders below are short Hitchhiker's Guide lines so the notebook runs; swap in your real passages.

In [21]:
# TODO (you): replace both with your own >=100-word passage and its translation.
#Source Wikipedia: Austrian Empire page
english_text = "The Austrian Empire,[b] officially known as the Empire of Austria, was a multinational European great power from 1804 to 1867, created by proclamation out of the realms of the Habsburgs. During its existence, it was the third most populous nation in Europe after the Russian Empire and the United Kingdom, while geographically, it was the third-largest empire in Europe after the Russian Empire and the First French Empire. The empire was proclaimed by Francis II in 1804 in response to Napoleon's declaration of the First French Empire, unifying all Habsburg possessions under one central government. It remained part of the Holy Roman Empire until the latter's dissolution in 1806. It continued fighting against Napoleon throughout the Napoleonic Wars, except for a period between 1809 and 1813, when Austria was first allied with Napoleon during the invasion of Russia and later neutral during the first few weeks of the Sixth Coalition War. Austria and its allies emerged victorious in the war, leading to the Congress of Vienna, which reaffirmed the empire as one of the great powers of the 19th century."
foreign_text = "Das Kaisertum Österreich , offiziell Kaiserreich Österreich , war von 1804 bis 1867 eine multinationale europäische Großmacht , die durch eine Proklamation aus den Gebieten der Habsburger hervorging. Während seiner Existenz war es nach dem Russischen Reich und dem Vereinigten Königreich die drittbevölkerungsreichste Nation Europas und flächenmäßig nach dem Russischen Reich und dem Ersten Französischen Kaiserreich das drittgrößte Reich Europas. Das Kaiserreich wurde 1804 von Franz II. als Reaktion auf Napoleons Ausrufung des Ersten Französischen Kaiserreichs proklamiert und vereinigte alle habsburgischen Besitzungen unter einer Zentralregierung. Es blieb bis zur Auflösung des Heiligen Römischen Reiches im Jahr 1806 Teil dieses Reiches. Österreich kämpfte während der gesamten Napoleonischen Kriege gegen Napoleon , mit Ausnahme der Jahre 1809 bis 1813, als es zunächst während des Russlandfeldzugs mit Napoleon verbündet und später in den ersten Wochen des Sechsten Koalitionskrieges neutral war . Österreich und seine Verbündeten gingen aus dem Krieg als Sieger hervor, was zum Wiener Kongress führte , der das Kaiserreich als eine der Großmächte des 19. Jahrhunderts bestätigte."

print('English words:', len(english_text.split()))
print('Foreign words:', len(foreign_text.split()))

def report(label, text):
    print(f'{label:9s} | chars {len(text):4d} | GPT-4 {count_tokens(text, gpt4):4d} | GPT-4o {count_tokens(text, gpt4o):4d}')

report('English', english_text)
report('Foreign', foreign_text)

tax_gpt4  = count_tokens(foreign_text, gpt4)  / count_tokens(english_text, gpt4)
tax_gpt4o = count_tokens(foreign_text, gpt4o) / count_tokens(english_text, gpt4o)
print(f'\nMultilingual tax  GPT-4: {tax_gpt4:.2f}x   GPT-4o: {tax_gpt4o:.2f}x')
# TODO (you): one or two sentences interpreting these numbers for YOUR language pair.

# I chose to copy a wikipedia page about the austrian empire and translate it to german, the english version has 14
# additional words, however it has 80 less characters than the german translation. This lines up with what I expected
# German tends to consolidate multiple words into one larger word which is why it has a higher character count despite 
# having less words. I took a look at some of the passages that my classmates translated and it seems like atleast for this
# Specific passage, the token ratio of german being 1.46x actually isn't that high for a foreign language.

English words: 180
Foreign words: 166
English   | chars 1109 | GPT-4  226 | GPT-4o  225
Foreign   | chars 1189 | GPT-4  329 | GPT-4o  267

Multilingual tax  GPT-4: 1.46x   GPT-4o: 1.19x


## Part 3: Evaluate with real figures

Turn the counts into engineering consequences. The skeleton below computes both; keep it pointed at your real passages.

In [22]:
CTX = 128_000
en = count_tokens(english_text, gpt4)
fo = count_tokens(foreign_text, gpt4)
print(f'A {CTX:,}-token window holds about {CTX//en:,} English copies and {CTX//fo:,} foreign copies of your passage.')
print(f'Per-request cost multiplier for the foreign language: {fo/en:.2f}x (billing is per token).')
# TODO (you): state what this means for a product serving users in your chosen language.

# This result indicates that a 128,000 token window is able to hold less copies of German texts over english texts.
# Additionally the per request multiplier being 1.46x means german translation requests cost almost 50% more tokens than english.
# This all indicates that for a product, this is much more geared towards english users, german users should expect higher
# costs and less text passages stored when using this software, any german company using this needs to keep that in mind when using this product.
# Additionally the difference in copy count for a 128,000 token window (566 for English and 389 for German) shows the massive
# Disparity between the token consumption of the two languages here, whether it is consistent across all German-English
# passage translations would require more testing but I think the 177 copy difference showcases that German is not as well
# optimized token wise as English is.

A 128,000-token window holds about 566 English copies and 389 foreign copies of your passage.
Per-request cost multiplier for the foreign language: 1.46x (billing is per token).


## Part 4: Bias splits and one failure

**TODO (you):** (a) pick three words where your non-English form fragments far worse than the English equivalent, and show both with `show_split`; (b) find ONE input whose token count defies intuition and explain it. A few failure candidates are demonstrated below to get you started; replace them with your own find and write the explanation plus a mitigation.

In [23]:
# (a) TODO (you): three real bias pairs from your languages.
print('English baselines:')
for w in ['Empire', 'European', 'United']:
    show_split(w)

print('\nGerman baselines:')
for w in ['Kaiserreich', 'Europäische', 'Vereinigte']:
    show_split(w)
    
print('\n(b) failure candidates to explore (replace with your own find):')

print('\n Token count that defies intuition')
show_split('French')
show_split('Französischen')

# TODO (you): explain WHY your chosen case behaves this way, and how you would budget or normalize around it.
# GPT 4 is more familiar with english words and is better able to group segments of english words together into tokens,
# Meanwhile the larger german words with more letters are segmented much more into tokens. With the german words, Europäische and Vereinigte,
# translating to european and united, both use four tokens as opposed to the respective english versions of both words only using one token,
# and empire using two tokens and kaiserreich using three tokens, this happens because of the way the words are split 
# into subsegments for tokenization. Because of this difference in word structure I would get samples of how many tokens 
# are used in total on english vs german sentences to get an idea of how to budget properly, as measuring by word or 
# character count is inconsistent when the tokenizer shows preference for larger english tokens.

# As for the failure case, Französischen gets divided into 5 tokens, despite only being 1 word, and multiple tokens only containing a single letter
# This is unintuitive as other longer words in english would not be subdivided so much, and single letters for whole tokens is very inefficient.
# To mitigate this, my recommendation here is the same as I explained above, budgeting it by token rather than by word or character count.
# I also wonder if the alternate characters that German uses, such as ö, would regularly be tokenized individually as it was in this word
# I believe it would and in which case that only strengthens the case for measuring by token and not by word count or character count.
# This is because German uses a lot of alternative characters that GPT 4 might not be the most familiar with and thus individually tokenize,
# so it would be very important to factor that in when trying to set a budget using this software.



English baselines:
'Empire'           -> 2 token(s): ['Emp', 'ire']
'European'         -> 1 token(s): ['European']
'United'           -> 1 token(s): ['United']

German baselines:
'Kaiserreich'      -> 3 token(s): ['K', 'aiser', 'reich']
'Europäische'      -> 4 token(s): ['Eu', 'rop', 'ä', 'ische']
'Vereinigte'       -> 4 token(s): ['V', 'ere', 'in', 'igte']

(b) failure candidates to explore (replace with your own find):

 Token count that defies intuition
'French'           -> 1 token(s): ['French']
'Französischen'    -> 5 token(s): ['Fran', 'z', 'ö', 'sis', 'chen']


## Part 5: Submit

Before you open the pull request, check:

- The notebook runs top to bottom on **your** passages, not the placeholders.
- Your three bias splits are shown and explained.
- The failure case has a cause and a mitigation.
- The PR description has a one-paragraph result summary with your headline numbers.
- You linked one issue in your repo logging this as a research note (title, inputs, what you found).

Rubric: repo quality (15), counts from both tokenizers (20), tax computed (15), three bias splits (20), cost and context figures (15), the failure case (10), PR hygiene (5).